# Penguins

### Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import pandas as pd
from matplotlib import pyplot as plt

data_path = "../data/penguins_size.csv"

penguins = pd.read_csv(data_path).dropna()
adelie = penguins[penguins['species'] == 'Adelie']

n = len(adelie)
sample_mean = adelie['flipper_length_mm'].mean()
sample_sd = adelie['flipper_length_mm'].std()
sigma_sq = sample_sd**2

# Prior: μ ~ N(200, 30²)
prior_mean = 200
prior_sd = 30
prior_precision = 1 / (prior_sd**2)

# Likelihood precision
likelihood_precision = n / sigma_sq

# Posterior parameters
posterior_precision = prior_precision + likelihood_precision
posterior_var = 1 / posterior_precision
posterior_sd = np.sqrt(posterior_var)
posterior_mean = (prior_precision * prior_mean + 
                  likelihood_precision * sample_mean) / posterior_precision

### Formal Hypothesis Statement

H₀: μ ∈ [200, 220]  (The average flipper length is between 200-220 mm)
Hₐ: μ ∉ [200, 220]  (The average flipper length is outside 200-220 mm)

Note: This is a two-sided (interval) hypothesis test because Hₐ includes
      values both below 200 mm AND above 220 mm.

### Part (b): Decision Using 95% Credible Interval

95% Credible Interval for μ: (189.05, 191.16)
Hypothesized interval H₀: [200, 220]

Decision analysis:
The credible interval lies COMPLETELY outside H₀
    Evidence against H₀; reject H₀ in favor of Hₐ

In [2]:
# Calculate 95% credible interval
ci_lower = stats.norm.ppf(0.025, posterior_mean, posterior_sd)
ci_upper = stats.norm.ppf(0.975, posterior_mean, posterior_sd)

print(f"95% Credible Interval for μ: ({ci_lower:.2f}, {ci_upper:.2f})")
print(f"Hypothesized interval H₀: [200, 220]")

overlap_lower = max(ci_lower, 200)
overlap_upper = min(ci_upper, 220)
overlap_exists = overlap_upper > overlap_lower

credible_inside_H0 = (ci_lower >= 200) and (ci_upper <= 220)
H0_inside_credible = (200 >= ci_lower) and (220 <= ci_upper)

print(f"\nSpecific check:")
print(f"    Is {ci_lower:.1f} ≥ 200: {ci_lower >= 200}")
print(f"    Is {ci_upper:.1f} ≤ 220: {ci_upper <= 220}")
print(f"        Since both are {ci_lower >= 200 and ci_upper <= 220}, the entire")
print(f"        credible interval {'IS' if credible_inside_H0 else 'IS NOT'} inside H₀")

95% Credible Interval for μ: (189.05, 191.16)
Hypothesized interval H₀: [200, 220]

Specific check:
    Is 189.0 ≥ 200: False
    Is 191.2 ≤ 220: True
        Since both are False, the entire
        credible interval IS NOT inside H₀


### Part (c): Posterior Probability of H₀

P(H₀ is true | data) = P(200 ≤ μ ≤ 220 | data) = 0.0000
                      = 0.00%

P(Hₐ is true | data) = 1.0000 (100.00%)

Interpretation:
  Strong evidence against H₀ (posterior probability < 25%)

Bayesian interpretation avoids p-values:
  - P(H₀ true | data) = 0.00%
  - P(Hₐ true | data) = 100.00%
  - This directly answers: 'What's the probability the hypothesis is true?'
    (Unlike frequentist p-values which answer a different question)

In [3]:
# P(μ ∈ [200, 220] | data)
prob_H0_given_data = stats.norm.cdf(220, posterior_mean, posterior_sd) - \
                     stats.norm.cdf(200, posterior_mean, posterior_sd)

# Also calculate complementary probability for Hₐ
prob_Ha_given_data = 1 - prob_H0_given_data

print(f"P(H₀ true | data) = {prob_H0_given_data:.2%}")
print(f"P(Hₐ true | data) = {prob_Ha_given_data:.2%}")

P(H₀ true | data) = 0.00%
P(Hₐ true | data) = 100.00%


### Part (d): SUMMARY OF EVIDENCE:
- Posterior mean μ = 190.1 mm
- 95% Credible Interval = (189.0, 191.2)
- P(200 ≤ μ ≤ 220 | data) = 0.00%
- H₀ interval [200, 220] is not entirely within CI

### CONCLUSION:
  The hypothesis H₀ is unlikely to be true (posterior probability = 0.00%).
  The evidence suggests the true mean flipper length falls outside the
  [200, 220] range.